# Interactive GRIB exploration

Run top to bottom (Shift+Enter per cell). Requires the kernel to use
`gribsvc/.venv` (numpy, pillow, pygrib). Start the kernel from the `gribsvc/`
directory so `from app import ...` resolves and `testdata/` is found.

The file-listing cell prints each file's grid extent. Nearest-gridpoint lookups
for points outside the grid snap to the nearest edge.

In [ ]:
# setup - make `app` importable and point at the data dir regardless of cwd
import os
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "testdata").is_dir() and (HERE / "gribsvc" / "testdata").is_dir():
    HERE = HERE / "gribsvc"  # notebook launched from repo root
sys.path.insert(0, str(HERE))
os.environ["GRIB_DATA_DIR"] = str(HERE / "testdata")

import numpy as np
import pygrib

from app import grib, render, sources
print("using", HERE)

In [ ]:
# what files / fields are available
files = sources.list_files()
print("available files:")
for i, f in enumerate(files):
    print(f"  [{i}] {f}")

# choose which file to explore (default: the last one — the whole-Finland grid)
path = sources.resolve(files[-1])
print("\nusing:", path.name)
params = grib.list_params(path)
v0, la0, lo0, _ = grib.field_grid(path, params[0]["param"], None)
print(f"messages: {len(params)} | grid {v0.shape} | "
      f"lat {la0.min():.2f}->{la0.max():.2f} lon {lo0.min():.2f}->{lo0.max():.2f}")
for p in params[:3]:
    print(p)

In [ ]:
# open the file directly with pygrib and inspect one message
grbs = pygrib.open(str(path))
msg = grbs[1]  # 1-indexed
print(msg)
print("shortName:", msg.shortName, "| name:", msg.name, "| units:", msg.units)
print("validDate:", msg.validDate, "| level:", msg.level, msg.typeOfLevel)

values = msg.values            # 2D numpy (masked) array
lats, lons = msg.latlons()
print("grid shape:", values.shape)
print("lat range:", float(lats.min()), "->", float(lats.max()))
print("lon range:", float(lons.min()), "->", float(lons.max()))
print("value range:", float(values.min()), "->", float(values.max()))

In [ ]:
# list every GRIB key on a message (great for discovering metadata)
for key in sorted(msg.keys()):
    try:
        print(f"{key} = {msg[key]}")
    except Exception:
        pass
grbs.close()

In [ ]:
# point extraction via the service helper (Kelvin -> Celsius for 2t)
pts = grib.extract_points(path, "2t", [
    (60.17, 24.94),  # Helsinki
    (61.50, 23.76),  # Tampere
    (66.50, 25.73),  # Rovaniemi
], None)
for pt in pts["points"]:
    c = None if pt["value"] is None else round(pt["value"] - 273.15, 2)
    print(f"({pt['lat']},{pt['lon']}) -> {c} C  @grid ({pt['grid_lat']:.2f},{pt['grid_lon']:.2f})")

In [ ]:
# forecast for the NEXT HOUR from now: pick the timestep nearest to now+1h.
from datetime import datetime, timezone, timedelta

times_2t = [p["valid_time"] for p in params if p["param"] == "2t"]
parsed = [grib.parse_time(t) for t in times_2t]   # naive UTC, like pygrib
now = datetime.now(timezone.utc).replace(tzinfo=None)
target = (now + timedelta(hours=1)).replace(minute=0, second=0, microsecond=0)

idx = min(range(len(parsed)), key=lambda i: abs((parsed[i] - target).total_seconds()))
at = parsed[idx]
print(f"now           {now:%Y-%m-%d %H:%MZ}")
print(f"next hour     {target:%Y-%m-%d %H:%MZ}")
print(f"using step    {times_2t[idx]}  (idx {idx} of {len(times_2t)})")

vals, vlats, vlons, meta = grib.field_grid(path, "2t", at)
celsius = vals - 273.15
print("min/mean/max C:", round(float(celsius.min()), 1),
      round(float(celsius.mean()), 1), round(float(celsius.max()), 1))

In [ ]:
# render a tile over the grid's coverage and view it inline
# Size height to the bbox's Web-Mercator aspect so the tile isn't stretched
# (this strip is wide+short, so a square would smear features vertically).
from app.render import _lon_to_merc_x, _lat_to_merc_y

bbox = (19.1, 59.7, 29.8, 61.5)
width = 768
aspect = (_lat_to_merc_y(bbox[3]) - _lat_to_merc_y(bbox[1])) / (
    _lon_to_merc_x(bbox[2]) - _lon_to_merc_x(bbox[0])
)
height = round(width * aspect)
png = render.render_png(vals, vlats, vlons, bbox, width, height, colormap="jet")
out_png = HERE / "tile.png"
out_png.write_bytes(png)
print("wrote", out_png, width, "x", height, "|", len(png), "bytes")

from IPython.display import Image as IPyImage
IPyImage(data=png)

### Optional: nicer heatmap with a colorbar
Install matplotlib into the venv first: `.venv/bin/pip install matplotlib`, then run the cell below.

In [ ]:
import matplotlib.pyplot as plt
# origin from the latitude order (this file scans south->north, so "lower")
origin = "lower" if vlats[0, 0] < vlats[-1, 0] else "upper"
plt.figure(figsize=(12, 4))
plt.imshow(celsius, origin=origin, cmap="jet", aspect="auto",
           extent=[float(vlons.min()), float(vlons.max()),
                   float(vlats.min()), float(vlats.max())])
plt.colorbar(label="C")
plt.title(f"2t @ {meta['valid_time']}")
plt.xlabel("lon"); plt.ylabel("lat")
plt.show()

### Overlay on the map of Finland (cartopy)
Coastlines + borders over the temperature field for geographic context.

In [ ]:
# overlay the field on the full map of Finland (cartopy)
# `.venv/bin/pip install cartopy` first; first run downloads Natural Earth data
# (needs internet). The map is zoomed to all of Finland; the temperature patch
# only fills the part this file actually covers.
#
# NOTE: Jupyter's inline backend crops figures with bbox_inches='tight' by
# default, which can eat a map and leave only the colorbar. Disable it here.
ip = get_ipython()
if ip is not None:
    ip.run_line_magic("config", "InlineBackend.print_figure_kwargs = {'bbox_inches': None}")

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# White background + black text so the title / axis labels / colorbar stay
# legible even when a dark IDE theme injects light-colored text defaults.
plt.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "text.color": "black", "axes.labelcolor": "black", "axes.edgecolor": "black",
    "axes.titlecolor": "black", "xtick.color": "black", "ytick.color": "black",
})

data_ext = [float(vlons.min()), float(vlons.max()), float(vlats.min()), float(vlats.max())]
map_ext = [18.0, 32.5, 59.0, 70.5]   # all of Finland; widen/narrow to taste

# This GRIB scans south->north (row 0 = southernmost). imshow's extent puts
# min lat at the bottom, so use origin="lower" to keep the field upright.
# (Derive it from the data rather than hardcoding.)
origin = "lower" if vlats[0, 0] < vlats[-1, 0] else "upper"

fig = plt.figure(figsize=(7, 8), facecolor="white")
ax = fig.add_axes([0.06, 0.04, 0.80, 0.84], projection=ccrs.PlateCarree())
ax.set_extent(map_ext, crs=ccrs.PlateCarree())

# base map so the country shape reads even outside the data coverage
ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="#cfe6f2")
ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#f2efe9")
# ax.add_feature(cfeature.LAKES.with_scale("50m"), facecolor="#cfe6f2")  # optional

im = ax.imshow(celsius, origin=origin, extent=data_ext, transform=ccrs.PlateCarree(),
               cmap="jet", alpha=0.85, zorder=3)
ax.coastlines(resolution="50m", linewidth=0.8, color="black", zorder=4)
ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.6, zorder=4)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)
gl.top_labels = gl.right_labels = False
gl.xlabel_style = {"color": "black"}
gl.ylabel_style = {"color": "black"}

cax = fig.add_axes([0.88, 0.20, 0.02, 0.60])
cb = fig.colorbar(im, cax=cax)
cb.set_label("C", color="black")
cb.ax.tick_params(colors="black")

# title (what + when) as a figure-level title in the top margin, where the
# cramped GeoAxes can't hide it.
field_name = next((p["name"] for p in params if p["param"] == "2t"), "2t")
when = meta["valid_time"].replace("T", " ").replace(":00Z", " UTC")
fig.suptitle(f"{field_name} (C) - {when}", color="black", fontsize=13, y=0.95)
plt.show()